In [8]:
# Semantic search

!pip install sentence-transformers
!pip install pymupdf
!pip install boto3 botocore



In [10]:
import os
import fitz
from sentence_transformers import SentenceTransformer, util
import urllib.parse, boto3
from botocore.config import Config
from botocore import UNSIGNED



In [3]:
model = SentenceTransformer('ai-forever/sbert_large_nlu_ru')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [11]:
from google.colab import drive
drive.mount('/content/drive')

folder_path = "/content/drive/MyDrive/russian_books"

def read_file(file_path):
    if file_path.endswith('.pdf'):
        text = ""
        with fitz.open(file_path) as doc:
            for page in doc:
                text += page.get_text()
        return text
    elif file_path.endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    return ""


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

file_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(('.pdf', '.txt'))]
file_texts = [read_file(f) for f in file_paths]

file_embeddings = model.encode(file_texts, convert_to_tensor=True)

query = input("Can you enter your question")
query_embedding = model.encode(query, convert_to_tensor=True)

cos_scores = util.pytorch_cos_sim(query_embedding, file_embeddings)
best_match_idx = int(cos_scores.argmax())

print(f"Best matching file: {file_paths[best_match_idx]}")
